[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/05_result_interpretation/05_analysis_viz.ipynb)


# 05 — 결과 해석·시각화

> 본문: [`05_result_interpretation.md`](05_result_interpretation.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 **여러분이 직접 돌린 결과**(`my_run/`)에서 계산합니다 — 설계 셀을 건너뛰면 저장소의 레퍼런스 결과로 이어져요.
> **분석 셀 실행 6초.**

> **① 직접 설계 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 진행합니다.
> 설계 셀은 NVIDIA GPU 에서 동작해요 — Colab 이면 **런타임 → 런타임 유형 변경 → T4 GPU**.
> 건너뛰어도 됩니다: 그러면 저장소에 커밋된 레퍼런스 결과로 이어집니다.

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 노트북은 **Colab과 로컬 모두**에서 동작합니다.

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`05_result_interpretation`) 폴더로 자동 이동한 뒤, `data/` 의 실제 결과로 실습합니다.
- **로컬**: 이 노트북을 `05_result_interpretation/` 폴더 안에서 열었다면 클론 없이 그대로 진행됩니다.

> 런타임은 **기본값 그대로** 두면 됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "05_result_interpretation"
PIP_PKGS = "pandas matplotlib"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# 필요한 라이브러리 확보: import 안 되는 것만 설치(Colab 새 런타임/로컬 모두 자동 대응)
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# ── 내가 만든 결과 우선, 없으면 레퍼런스 ──────────────────────────────────────
#   MY  : 이 노트북에서 내가 직접 돌려 만든 산출물
#   REF : 저장소에 커밋된 레퍼런스 결과(대조군 · 실행을 건너뛰어도 실습이 이어지도록)
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """산출물 하나를 찾는다 — 내가 만든 my_run/ 트리를 먼저 뒤지고, 없으면 레퍼런스 폴더에서.

    파일명 규칙이 달라도(내 실행 rank1_*.cif / 레퍼런스 rank001_*.cif,
    final_<budget>_designs / final_designs) 같은 글롭으로 잡히도록 설계했습니다.
    """
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 이미 설계를 돌렸다면 그 결과를 이어받는다(이 챕터에서 다시 안 돌려도 됨).

    내 my_run/ 에 결과가 있으면 그대로 쓰고, 없으면 인자로 준 순서대로 앞 챕터를 찾는다.
    아무 데도 없으면 MY 를 그대로 둬서 find_one() 이 레퍼런스로 폴백하게 한다.
    """
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 내가 만든 그림은 my_ 접두어로 저장 — 저장소에 커밋된 레퍼런스 그림을 덮어쓰지 않도록.
def my_fig(name):
    return f"my_{name}"

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd())

## 1) 직접 돌려보기 — 내 결과 만들기

아래 셀이 **BoltzGen 을 실제로 실행**해 `my_run/` 에 내 설계 결과를 만듭니다.
Colab 이면 **런타임 → 런타임 유형 변경 → T4 GPU** 로 바꾼 뒤 실행하세요.

- 학습용으로 `num_designs=8 --budget=4` 규모입니다(레퍼런스 결과는 num_designs=100).
- 소요 시간: **약 10분**(실측 585초, 최종 4개) — 가중치가 이미 캐시된 상태 기준이에요.
- 첫 실행은 모델 가중치(~6GB) 다운로드가 더해져 더 걸립니다.
- **이 셀을 건너뛰어도 됩니다** — 그러면 아래 분석이 저장소의 레퍼런스 결과로 이어집니다.

In [ ]:
SPEC, PROTOCOL = "example/vanilla_protein/1g13prot.yaml", "protein-anything"
NUM_DESIGNS, BUDGET = 8, 4

import shutil
OUT = MY.resolve()

def _gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return shutil.which("nvidia-smi") is not None

if not _gpu():
    print("GPU 런타임이 아니라 설계 실행을 건너뜁니다.")
    print("  · Colab: [런타임 → 런타임 유형 변경 → T4 GPU] 후 이 셀을 다시 실행하세요.")
    print("  · 그대로 진행해도 됩니다 — 아래 분석은 레퍼런스 결과로 이어집니다.")
else:
    SRC = ADV_ROOT / ".boltzgen_src"          # 예제 명세·타깃 CIF 가 들어 있는 BoltzGen 소스
    if not SRC.exists():
        _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')
    if not _have("boltzgen"):
        _run(f'"{sys.executable}" -m pip -q install -e "{SRC}"')
    try:
        _run(f'cd "{SRC}" && boltzgen run {SPEC} --output "{OUT}" --protocol {PROTOCOL} '
             f'--num_designs {NUM_DESIGNS} --budget {BUDGET}')
        print("\n내 결과 →", OUT / "final_ranked_designs")
    except Exception as e:
        # 실행이 중간에 죽어도 노트북은 계속 갑니다(완료된 스텝 산출물은 my_run/ 에 남아요).
        print("\n설계 실행이 도중에 멈췄어요:", type(e).__name__)
        print("  · 이 셀을 그대로 다시 실행하세요 — 같은 --output 폴더의 산출물을 재사용해 이어서 끝냅니다.")
        print("    (실측: 신규 실행 763초 → 중단 후 재개 486초)")
        print("  · GPU 메모리가 부족했다면 규모를 줄여보세요: NUM_DESIGNS 를 4, BUDGET 을 2 로.")
        print("  · 그대로 두고 아래로 내려가도 됩니다 — 레퍼런스 결과로 실습이 이어집니다.")

In [ ]:
from boltzgen_viz import load_metrics, metrics_overview, compare_bars
import pandas as pd
# Ch.04 에서 스모크 실행을 했다면 그 결과를 이어받습니다(05에서 다시 안 돌려도 돼요)
inherit_run("../04_basic_usage/my_run")
CSV = str(find_one("final_designs_metrics_*.csv", "data/vanilla"))
df = pd.read_csv(CSV)
print("디자인:", len(df), "| 전체 컬럼:", df.shape[1], "(메트릭 240여 종)")

## 1) 핵심 메트릭 7군 (본문 5.1~5.7)
신뢰도(pTM/ipTM) · 위치오차(PAE) · 구조편차(RMSD) · 인터페이스(H-bond/ΔSASA) 를 한 표로.

In [ ]:
df[cols_in(df, "id", "final_rank", "design_ptm", "design_to_target_iptm",
           "min_design_to_target_pae", "filter_rmsd", "plip_hbonds_refolded",
           "delta_sasa_refolded")].sort_values("final_rank")

## 2) 2×2 메트릭 개요 그래프 (스타일 매칭)
pTM(보라)·ipTM(주황)·RMSD(청록) 바 + 길이↔H-bond 산점도. 임계선: pTM 0.7 / ipTM 0.5 / RMSD 2.0Å.

> 내가 만든 그림은 `my_05_vanilla_metrics.png` 로 저장돼요(본문에 실린 레퍼런스 그림을 덮어쓰지 않도록).

In [ ]:
rows = load_metrics(CSV)
FIG = my_fig("05_vanilla_metrics.png")
metrics_overview(rows, "Vanilla Protein — Design Metrics Overview", FIG)
from IPython.display import Image; Image(FIG)

## 3) 메트릭 간 상관관계
각 메트릭이 독립적 정보를 주는지 확인 → 하나만 보지 말고 종합 판단.

> 직접 돌린 결과(4~8개)는 표본이 작아 상관계수가 불안정해요 — 경향만 보세요.

In [ ]:
m = df[cols_in(df, "design_ptm", "design_to_target_iptm", "filter_rmsd",
                "plip_hbonds_refolded", "delta_sasa_refolded")].astype(float)
m.corr().round(2)

## 4) 보조 분석 파일 (analysis 스텝 산출)
`aggregate_metrics_analyze.csv`(타깃 요약) / `per_target_metrics_analyze.csv`(타깃별).

In [ ]:
import pandas as pd
for name in ["aggregate_metrics_analyze.csv", "per_target_metrics_analyze.csv"]:
    try:
        p = find_one(name, "data/vanilla")
        a = pd.read_csv(p); print(p, "→", a.shape); display(a.head(3))
    except Exception as e:
        print(name, "건너뜀:", e)

## 5) 해석 요점 (본문 5.8~5.10) — 내 결과의 실제 수치로

읽는 원칙(어느 결과에나 적용).

- **ipTM**(`design_to_target_iptm`)이 결합의 핵심 지표 — 가장 높은 것부터 후보로 봄.
- **RMSD**(`filter_rmsd`)는 자기일관성 — 낮을수록 "설계한 모양이 서열로 재현"됨.
- **pTM** 이 높아도 ipTM 이 낮을 수 있어 **함께** 봐야 함(둘은 다른 걸 잼).
- 순위는 `rank_*` 들을 종합한 `final_rank`. pLDDT(`complex_plddt`)는 순위의 주지표가 아님.

아래 셀이 **여러분 데이터에서** 직접 뽑아 줍니다. 본문 5.10 의 구체적 수치
(rank 5·6·8·9·10 이 ipTM 0.6 이상 등)는 **레퍼런스 실행(num_designs=100, budget=10)** 기준이라,
직접 돌린 결과(4~8개)와는 rank 번호도 값도 다릅니다 — 표본이 작으니 당연해요.

In [ ]:
n = len(df)
print(f"내 결과: 디자인 {n}개 (레퍼런스는 10개 — rank 번호가 다른 게 정상)\n")
k = min(3, n)
print(f"ipTM 상위 {k}:")
print(df.nlargest(k, "design_to_target_iptm")[["final_rank","id","design_to_target_iptm"]].to_string(index=False))
print(f"\nRMSD 최저 {k}:")
print(df.nsmallest(k, "filter_rmsd")[["final_rank","id","filter_rmsd"]].to_string(index=False))
print("\npTM 최고:", df.loc[df.design_ptm.idxmax(), ["final_rank","id","design_ptm"]].to_dict())
print("\nipTM 0.6 이상:", sorted(df.loc[df.design_to_target_iptm >= 0.6, "final_rank"].tolist()) or "없음")